In [ ]:
from IPython.display import HTML, display

def set_css(*args, **kwargs):
    display(HTML('''
    <style>
        pre {
            white-space: pre-wrap;
        }
    </style>
    '''))
    
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
import re

def clean_text(s: str) -> str:
    s = re.sub(r'\[deleted\]|\[removed\]', '', s, flags=re.IGNORECASE)
    s = re.sub(r'&amp;?', '', s)
    s = s.replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

In [ ]:
%pip install faiss-cpu
%pip install convokit

In [ ]:
from convokit import Corpus
from dataclasses import dataclass
from typing import List
from functools import reduce
from pathlib import Path
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

model = SentenceTransformer("fashion-bert-output-v4")

In [ ]:
base = Path("fashion-corpora")
corpora = [Corpus(str(p)) 
           for p in base.iterdir() 
           if p.is_dir() and p.suffix == ".corpus"]


texts, ids = [], []
for c in corpora:
    #print()
    #print(c)
    for utt in c.iter_utterances():
        #print(utt)
        if utt.text and utt.text.strip():
            convo = utt.get_conversation()
            doc = utt.text
            if len(doc.split()) >= 5:
                texts.append(doc)
                ids.append(utt.id)


clean_texts = list(filter(None, [clean_text(t) for t in texts]))

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

def filter_distinctive_reviews(reviews, bottom_percentile=10):
    vectorizer = TfidfVectorizer(stop_words='english', min_df=2)
    tfidf_matrix = vectorizer.fit_transform(reviews)
    tfidf_scores = tfidf_matrix.mean(axis=1).A1
    threshold = np.percentile(tfidf_scores, bottom_percentile)
    filtered_reviews = [rev for rev, score in zip(reviews, tfidf_scores) if score > threshold]

    return filtered_reviews

In [ ]:
filtered_texts_reddit = filter_distinctive_reviews(clean_texts)

In [ ]:
def extract_product_texts_and_ids(data):
    p_texts, p_ids, p_metadata = [], [], []
    for item in data:
        product_id = item.get('ID')
        if not product_id:
            continue

        name = item.get('name', '')
        desc = item.get('description', '')
        clean_desc = clean_text(desc)
        if not clean_desc:
            continue

        p_texts.append(name + ': ' + clean_desc)
        p_ids.append(product_id)

    return p_texts, p_ids

In [ ]:
import json

json_path = Path("COMBINED-FINAL-DEDUPED-CLEAN2.json")
with json_path.open("r", encoding="utf-8") as f:
    records = json.load(f)

prod_meta = []
for item in records:
    prod_meta.append({
        "train_id": item.get("ID"),
        "name":     item.get("name", ""),
        "price":    item.get("price", ""),
        "category": item.get("category", "")
    })

product_texts, ids  = extract_product_texts_and_ids(records)
print('Size of text list: ' + str(len(product_texts)))
print(product_texts)

In [ ]:
#create the reddit embeddings...

reddit_embs  = model.encode(filtered_texts_reddit, convert_to_numpy=True, show_progress_bar=True)

norms = np.linalg.norm(reddit_embs, axis=1, keepdims=True)
embs  = reddit_embs / np.clip(norms, 1e-8, None)

dim   = reddit_embs.shape[1]
index = faiss.IndexFlatIP(dim)  
index.add(reddit_embs.astype("float32"))

In [ ]:
#create the product embeddings
prod_embs = []
prod_meta = []

for item in records:
    desc = clean_text(item.get("description", ""))
    if not desc:
        continue
    emb = model.encode(desc, convert_to_numpy=True)
    prod_embs.append(emb)
    prod_meta.append({
        "train_id": item["ID"],
        "name": item["name"],
        "price": item["price"],
        "category": item["category"]
    })

prod_embs = np.array(prod_embs)

In [ ]:
np.save("julia_tries_reddit_embs.npy", reddit_embs)
np.save("julia_tries_prod_embs.npy", prod_embs)

In [ ]:
print(prod_meta)

In [ ]:
import numpy as np

def cosine_similarity(a: np.ndarray, b: np.ndarray):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return np.dot(a_norm, b_norm.T)

In [ ]:
def search_products(query: str,
                    social_embs: np.ndarray,
                    product_embs: np.ndarray,
                    prod_meta: list[dict],
                    model,
                    k_corpus: int = 10,
                    k_prod: int = 20,
                    alpha: float = 1.0,
                    beta: float = 0.75):
    
    q_emb = model.encode([query], convert_to_numpy=True)
    q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)  # (1, dim)
    
    sim_u = cosine_similarity(q_emb, social_embs)  # (1, N)
    idxs_u = np.argsort(sim_u[0])[::-1][:k_corpus]
    top_u_embs = social_embs[idxs_u]
    
    expanded = alpha * q_emb + beta * top_u_embs.mean(axis=0, keepdims=True)
    expanded /= np.linalg.norm(expanded, axis=1, keepdims=True)
    
    sim_p = cosine_similarity(expanded, product_embs)  # (1, N)
    idxs_p = np.argsort(sim_p[0])[::-1][:k_prod]
    
    results = []
    for idx in idxs_p:
        if idx >= len(prod_meta):
            print("skipped")
            continue  # skip invalid index
        meta = prod_meta[idx]
        results.append({
            "train_id": meta["train_id"],
            "name":     meta["name"],
            "price":    meta["price"],
            "category": meta["category"],
            "score":    float(sim_p[0][idx])
        })
    return results


In [ ]:
print("Product embeddings:", prod_embs.shape[0])
print("Product metadata:  ", len(prod_meta))

In [ ]:
import numpy as np

# Load embeddings
r_embs = np.load("julia_tries_reddit_embs.npy")
p_embs = np.load("julia_tries_prod_embs.npy")

# Run search
results = search_products(
    "cutesy",
    r_embs,
    p_embs,
    prod_meta,
    model,
    k_corpus=10,
    k_prod=10
)

# Display results
for r in results:
    print(f"{r['score']:.3f}\t{r['name']} (${r['price']}) — {r['category']}")
